# Customer Churn — Data Quality and Exploratory Analysis

**Task 02 | Data Analytics Internship**

This notebook audits the supplied customer churn sample, documents cleaning decisions, validates business rules, and explores initial patterns. Run from this folder so the CSV paths resolve.

**Important:** The supplied sample has only 15 records. Findings are descriptive and must not be treated as population estimates or causal conclusions.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path(".")
raw = pd.read_csv(DATA_DIR / "customer_churn_raw.csv")
raw.head()


## 1. Structure and data types

In [ ]:
print("Shape:", raw.shape)
display(raw.info())
display(raw.describe(include="all").T)


## 2. Missing values and duplicates

In [ ]:
missing_audit = raw.isna().sum().rename("missing_values").to_frame()
duplicate_count = raw.duplicated().sum()
display(missing_audit)
print("Exact duplicate rows:", duplicate_count)


## 3. Cleaning and documented decisions

- Standardize column names to lowercase snake_case.
- Trim whitespace from text fields.
- Convert numeric fields explicitly.
- Keep all rows because no exact duplicates or missing values were found.
- Do not remove IQR-flagged values automatically; investigate them first.

In [ ]:
clean = raw.copy()
clean.columns = [c.strip().lower().replace(" ", "_") for c in clean.columns]
text_cols = clean.select_dtypes(include="object").columns
for col in text_cols:
    clean[col] = clean[col].astype("string").str.strip()

numeric_cols = ["age", "tenuremonths", "monthlycharges", "totalcharges", "supporttickets"]
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

clean = clean.drop_duplicates()
clean.to_csv(DATA_DIR / "cleaned_customer_churn.csv", index=False)
display(clean.head())
print("Cleaned shape:", clean.shape)


## 4. Numeric outlier screening using the 1.5×IQR rule

In [ ]:
outlier_rows = []
for col in numeric_cols:
    q1, q3 = clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = ((clean[col] < lower) | (clean[col] > upper)).sum()
    outlier_rows.append([col, q1, q3, lower, upper, int(count)])

outlier_audit = pd.DataFrame(outlier_rows, columns=[
    "column", "q1", "q3", "lower_fence", "upper_fence", "potential_outliers"
])
display(outlier_audit.round(2))
outlier_audit.to_csv(DATA_DIR / "iqr_outlier_audit.csv", index=False)


## 5. Business-rule validation

In [ ]:
# Check whether total charges reconcile with monthly charges × tenure.
clean["expected_totalcharges"] = (clean["monthlycharges"] * clean["tenuremonths"]).round(2)
clean["totalcharges_match"] = np.isclose(
    clean["totalcharges"], clean["expected_totalcharges"], atol=0.01
)
display(clean[["customerid", "totalcharges", "expected_totalcharges", "totalcharges_match"]])
print("Rows passing charge reconciliation:", int(clean["totalcharges_match"].sum()), "/", len(clean))


## 6. Summary statistics and churn distribution

In [ ]:
display(clean[numeric_cols].describe().round(2))
churn_counts = clean["churn"].value_counts()
churn_rate = clean["churn"].eq("Yes").mean() * 100
print(f"Sample churn rate: {churn_rate:.1f}%")
display(churn_counts)


## 7. Exploratory patterns

In [ ]:
# Churn count
clean["churn"].value_counts().reindex(["No", "Yes"]).plot(kind="bar", title="Customer churn distribution")
plt.xlabel("Churn"); plt.ylabel("Customers"); plt.tight_layout(); plt.show()

# Churn rate by contract type
contract_churn = clean.groupby("contracttype")["churn"].apply(lambda x: x.eq("Yes").mean() * 100)
display(contract_churn.rename("churn_rate_percent").to_frame())
contract_churn.reindex(["Month-to-Month", "One Year", "Two Year"]).plot(kind="bar", title="Churn rate by contract type")
plt.xlabel("Contract type"); plt.ylabel("Churn rate (%)"); plt.xticks(rotation=15); plt.tight_layout(); plt.show()

# Support ticket average by churn
support_summary = clean.groupby("churn")["supporttickets"].agg(["count", "mean", "median"])
display(support_summary)
support_summary["mean"].reindex(["No", "Yes"]).plot(kind="bar", title="Average support tickets by churn status")
plt.xlabel("Churn"); plt.ylabel("Average support tickets"); plt.tight_layout(); plt.show()


## 8. Findings, recommendations and limitations

### Findings
- The sample contains 15 customers: 7 churned and 8 did not (46.7% sample churn).
- All 7 Month-to-Month customers churned in this sample; none of the One Year or Two Year customers churned.
- Churned customers averaged 4.29 support tickets, compared with 1.00 among non-churned customers.
- All 15 total-charge values reconcile with monthly charges multiplied by tenure months within 0.01.

### Recommendations
Validate the Month-to-Month segment and review support interactions as initial investigation priorities. Obtain a larger time-based dataset before making business-wide decisions. Track churn by contract type and support-ticket volume over time.

### Limitations
Only 15 synthetic/sample records are available. The observed pattern may be an artifact of sample construction. These are descriptive associations, not causal effects. Currency and churn observation window are unspecified. No predictive model or population-level inference is appropriate.
